In [ ]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

print(f"La root del progetto è: {PROJECT_ROOT}")

In [ ]:
from paths import INV_PATH, TP1_PATH, MODEL_PATH, ROCK_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP1_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + ROCK_PATH

model_name = "Rock1.blend"
load_data_bound = ABS_PATH + "./files/data.csv"

fig_dir = "figures/"
loss_fig = ABS_PATH + fig_dir + "loss.png"
param_fig = ABS_PATH + fig_dir + "param.png"

params_save = ABS_PATH + "files/pred_parameters.csv"

In [ ]:
# Import
import torch
import pandas as pd
import matplotlib.pyplot as plt

from modelaquisition.bl2pina import Blend2Pina
from pina.solvers.pinns import RBAPINN
from pina.operators import laplacian
from pina.callbacks import MetricTracker
from pina.geometry import CartesianDomain
from pina.model import ResidualFeedForward
from pina import LabelTensor, Trainer, Plotter
from pina.condition import Condition, Equation
from pytorch_lightning.callbacks import Callback, StochasticWeightAveraging
from pina.problem import SpatialProblem, InverseProblem

In [ ]:
torch.set_default_dtype(torch.float64)

In [ ]:
# Network variables
lear_rate = 1e-3
swa_lr = 1e-4
decay_rt = 1e-8

# Solver variables
epochs=3_000
batch=50
acc_str = 'gpu'

ipt_var = 3
out_var = 1
lay = 2
neur = 400

# Num points
int_points = 500
bound_points = 200

In [ ]:
rock = Blend2Pina(LOAD_MODEL + model_name)

rock_int = rock.intern()
rock_bound = rock.boundary()

In [ ]:
df = pd.read_csv(load_data_bound, sep=";", index_col=0)

input_pts = df.iloc[:, :3].values
output_pts = df.iloc[:, -1].values

input_pts = LabelTensor(
    x=torch.tensor(input_pts, dtype=torch.float64),
    labels=['x', 'y', 'z']
)
output_pts = LabelTensor(
    x=torch.reshape(torch.tensor(output_pts, dtype=torch.float64), (output_pts.shape[0], 1)),
    labels=['u']
)

In [ ]:
class RockPoisson(SpatialProblem, InverseProblem):

    input_variables = ['x','y','z']
    output_variables = ['u']
    spatial_domain = rock_int
    # Definiamo il range per i parametri
    unknown_parameter_domain = CartesianDomain(
        {
            'lambda' : [0, 1],
            'alpha' : [0, 1],
            'beta' : [0, 1]
        }
    )

    # Residual
    @staticmethod
    def residual(input_, output_, params_):
        lap_u = laplacian(output_=output_, input_=input_, components=['u'], d=['x', 'y', 'z'])
        force_term = - (params_['alpha']**2 + params_['beta']**2) * (
            torch.pi**2 * params_['lambda'] * input_.extract('x') * 
            torch.cos(params_['alpha'] * torch.pi * input_.extract('y')) * 
            torch.sin(params_['beta'] * torch.pi * input_.extract('z'))
        )
        return lap_u - force_term
    
    def boundary(input_, output_, params_):
        ref = (
            params_['lambda'] * input_.extract('x') *
            torch.cos(params_['alpha'] * torch.pi * input_.extract('y')) * 
            torch.sin(params_['beta'] * torch.pi * input_.extract('z'))
        )
        return output_.extract('u') - ref
    
    conditions = {
        'Omega' : Condition(
            location=rock_int,
            equation=Equation(residual)
        ),
        'Gamma' : Condition(
            location=rock_bound,
            equation=Equation(boundary)
        ),
        'data' : Condition(
            input_points=input_pts.extract(['x','y','z']),
            output_points=output_pts
        )
    }

In [ ]:
problem = RockPoisson()

problem.discretise_domain(
    n=int_points,
    mode='random',
    locations=['Omega']
)

problem.discretise_domain(
    n=bound_points,
    mode='random',
    locations=['Gamma']
)

In [ ]:
class HardMLP(torch.nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__()
        self.layers = ResidualFeedForward(*args, **kwargs)

    # Nel metodo forward implementiamo il vincolo rigido
    def forward(self, x):
        return  self.layers(x)

In [ ]:
# directory temporanea per salvare i log del training
tmp_dir = ABS_PATH + "rock_poisson_inverse"

class SaveParameters(Callback):
    """
    Callback per salvare i parametri del modello ogni 100 epoche.
    """
    def on_train_epoch_end(self, trainer, _):
        if trainer.current_epoch % 100 == 99:
            torch.save(
                trainer.solver.problem.unknown_parameters,
                '{}/parameters_epoch{}'.format(tmp_dir, trainer.current_epoch)
            )

In [ ]:
# Modello
model = HardMLP(
    input_dimensions=ipt_var,
    output_dimensions=out_var,
    n_layers=lay,
    inner_size=neur
)
# Solver
pinn=RBAPINN(
    problem=problem,
    model=model,
    optimizer_kwargs={
        'lr' : lear_rate,
        'weight_decay' : decay_rt
    },
)

# Trainer
trainer=Trainer(
    solver=pinn,
    max_epochs=epochs,
    batch_size=None,
    accelerator=acc_str,
    precision='64-true',
    callbacks=[SaveParameters(), MetricTracker(), StochasticWeightAveraging(swa_lrs=swa_lr)]
)

# Addestramento
trainer.train()

In [ ]:
my_pl = Plotter()

my_pl.plot_loss(
    trainer=trainer,
    metrics=['Omega_loss'],
    label='Omega_loss',
    logy=True
)

my_pl.plot_loss(
    trainer=trainer,
    metrics=['Gamma_loss'],
    label='Gamma_loss',
    logy=True
)

my_pl.plot_loss(
    trainer=trainer,
    metrics=['data_loss'],
    label='data_loss',
    logy=True
)

plt.savefig(loss_fig, transparent=True)
plt.show()

In [ ]:
epochs_saved = range(99, epochs, 100)
parameters = torch.empty(
    size=(int(epochs/100), 3)
)

for i, epoch in enumerate(epochs_saved):
    params_torch = torch.load('{}/parameters_epoch{}'.format(tmp_dir, epoch))
    for e, var in enumerate(pinn.problem.unknown_variables):
        parameters[i, e] = params_torch[var].data

pred_alpha, pred_beta, pred_lambda = parameters[-1, :]

# Grafico dei parametri
plt.close()
plt.plot(epochs_saved, parameters[:, 2], label='lambda', marker='o')
plt.plot(epochs_saved, parameters[:, 0], label='alpha', marker='s')
plt.plot(epochs_saved, parameters[:, 1], label='beta', marker='^')
plt.ylim(0, 1)
plt.grid()
plt.legend()
plt.xlabel("Epochs")
plt.ylabel("Parameters")
plt.savefig(param_fig, transparent=True)
plt.show()

In [ ]:
par_lambda = torch.tensor(.1)
par_alpha = torch.tensor(.2)
par_beta = torch.tensor(.5)

err_rel_lambda = torch.norm(pred_lambda-par_lambda)/torch.norm(par_lambda)
err_rel_alpha = torch.norm(pred_alpha-par_alpha)/torch.norm(par_alpha)
err_rel_beta = torch.norm(pred_beta-par_beta)/torch.norm(par_beta)

print("RELATIVE ERRORS")
print(f"lambda: {err_rel_lambda.item(): .2e}")
print(f"alpha: {err_rel_alpha.item(): .2e}")
print(f"beta: {err_rel_beta.item(): .2e}")

In [ ]:
df = pd.DataFrame(
    data=[[pred_lambda.item()], [pred_alpha.item()], [pred_beta.item()]],
    columns=["predictions"],
    index=["lambda", "alpha", "beta"]
)

df.to_csv(params_save, sep=";")

In [ ]:
torch.save(model, ABS_PATH + f"models/model_L{lay}_N{neur}_EP{epochs}.pth")